# به نام خدا

### **06-01**

<div dir='rtl'>

به زبان ساده : EDA (تحلیل اکتشافی داده‌ها) به معنای درک داده‌ها پیش از هرگونه فرضیه‌سازی یا به‌کارگیری مدل‌های پیچیده است.

مثل اینکه مریضی اومده پیشتون و تازه می خوای بفهمی که چشه.

- **7 مرحله اصلی EDA:**
1. Descriptive Statistics کل داده چه آماری داره؟
2. Data Distribution Analysis تک تک هر داده خصوصا داده هدف چه توزیعی دارد
3. Correlation Analysis ارتباط بین دو یا چند متغیر چجوریاس؟ مخالف یا وابسته به هم اند یا هیچ تاثیری رو هم ندارن؟
4. Grouping and Aggregation
5. Time Series Analysis (if applicable)
6. Handling Missing Data and Outliers
7. Feature Engineering Insights

- **چرخه کاری معمول در علم داده: (عقب و جلو زیاد داره)**
1. Problem Definition
2. Data Collection
3. Data Cleaning and Preparation
4. Exploratory Data Analysis
5. Feature Engineering
6. Modeling
7. Evaluation and Interpretation
8. Deployment and Monitoring

- نکته:

کنار مصورسازی، عدد هم داشته باش

---

### **06-02**
<div dir='rtl'>

**1 اولین کار بعد از گرفتن داده، ببین نوعش چیه.**

انواع داده:

1. Numeric
2. String
3. Boolean
4. Date and Time
</div>

- `df.dtypes`
- `df['col'].dtype`
- `df.info()`
<div dir='rtl'>

**2 برسی صحت ستون های عددی**
- NaN داره؟
- اگر string تبدیل کن به عدد : `pd.to_numeric`
</div>

- `df.describe()`
<div dir='rtl'>

**3 بررسی مقادیر یکتا در ستون‌های دسته‌ای**
</div>

- `df.['col'].unique()`
<div dir='rtl'>

**4 شناسایی انواع داده‌های ترکیبی | Mixed Data Types**
</div>

```python
# برای تشخیص دیتافریم هایی که ستون هایی با داده های میکس دارند
def detect_mixed_types(df): # ورودی، خود دیتافریم رو میگیره
    mixed_types = {}
    for column in df.columns:
        types = df[column].apply(lambda v: type(v).__name__).value_counts()
        if len(types) > 1:
            mixed_types[column] = types.to_dict()
    return mixed_types
```

```python
# برای ستون‌هایی که باید عددی باشند اما حاوی مقادیر رشته‌ای هستند
def clean_numeric_column(series): # ورودی ستون میگیره
    # Convert to numeric, forcing errors to NaN
    numeric_series = pd.to_numeric(series, errors="coerce")

    # Check what percentage of data was lost
    percent_lost = (series.dropna().size - numeric_series.count()) / series.size * 100

    print(f"Percentage of data lost: {percent_lost:.2f}%")
    return numeric_series
```
<div dir='rtl'>

**5 انتخاب براساس نوع داده**
- مثلا اگر بخوای عملیاتی رو روی یک داده خاص انجام بدی
</div>

- `df.select_dtypes(include=[""])`
<div dir='rtl'>

**6 برسی میزان مصرف حافظه**
</div>

- `df.memory_usage()`
<div dir='rtl'>

**7 میتونی قبل از بارگزاری دیتافریم رو تغییر بدی | Ensuring Data Integrity During Loading**

**8 range دادها**
- مثال: رنج سن از یک حدی زیاد یا کم نباشه - اگر داده منفی داریم مثبتش کن

**9 بررسی رکوردهای تکراری | Checking for Duplicate Records**

**10 تأیید یکپارچگی داده‌های طبقه‌بندی‌شده**
- داده های دسته بندی شده درست دسته بندی شده اند؟
</div>

- `df["col"].isin()`
---

### **06-02**
<div dir='rtl'>

مراحل کلی Handling Missing Data:
1. انواعش چیاست؟
2. تشخیص بدیم
3. پرش کن

نکته:
- درک ماهیت Missing Data مهمتر از اینکه یک الگوریتم خاصی برای پر کردنش بنویسی

**مرحله 2**

تعداد و درصد داده گمشده در هر ستون:
</div>

```python
# Count missing values in each column
missing_counts = df.isnull().sum()

# Calculate percentage of missing values
missing_percentages = 100 * df.isnull().sum() / len(df)

# Combine counts and percentages in a single dataframe
missing_table = pd.concat(
    [missing_counts, missing_percentages],
    axis=1,
    keys=["Missing Count", "Missing Percentage"],
)
missing_table
```
<div dir='rtl'>
تصویری: (تعداد feature زیاد باشه این خوب نیست)
</div>

```python
msno.matrix(df)
```
<div dir='rtl'>

Detecting Hidden Missing Values :

بعضی وقتا مقدارگمشده دقیقا NaN نیست و شکل دیگه ای دارند. (معنیشون ناله ولی دقیقا NaN نیستند)

- پیداشون کن:
</div>

```python
# Check for suspicious values that might represent missing data
suspicious_values = ["N/A", "None", "-999", "", " "]

for column in df.columns:
    if df[column].dtype == "object":  # Only check string columns
        value_counts = df[column].value_counts()
        for value in suspicious_values:
            if value in value_counts.index:
                print(
                    f"Column '{column}' contains {value_counts[value]} instances of '{value}'"
                )
```
<div dir='rtl'>

- جایگزین کن:
</div>

```python
# Convert suspicious values to NaN
df.replace(suspicious_values, np.nan)
```
<div dir='rtl'>

درصد داده های گم شده: 
</div>

```python
def categorize_missingness(df, threshold=5):
    missing_percentages = 100 * df.isnull().sum() / len(df)

    no_missing = missing_percentages[missing_percentages == 0].index.tolist()
    low_missing = missing_percentages[
        (missing_percentages > 0) & (missing_percentages <= threshold)
    ].index.tolist()
    high_missing = missing_percentages[missing_percentages > threshold].index.tolist()

    print(f"Variables with no missing data: {no_missing}")
    print(f"Variables with low missing data (<= {threshold}%): {low_missing}")
    print(f"Variables with high missing data (> {threshold}%): {high_missing}")


categorize_missingness(df)
```
<div dir='rtl'>

**مرحله 3**

1. با میانگین یا median پر کن

2. برای دیتاهای گروه بندی با mode :
</div>

```python
df["col_mode_imputed"] = df["col"].fillna(
    df["col"].mode()[0]
)

print("Original 'col' value counts:")df
print(df["col"].value_counts(dropna=False))
print("\n'embarked' value counts after mode imputation:")
print(df["col_mode_imputed"].value_counts(dropna=False))
```
<div dir='rtl'>

3. Regression Imputation
- نکته: بعد از این کار همبستگی بین feature ها بیشتر میشه

4. K-Nearest Neighbors (KNN) Imputation

5. Time Series Imputation
</div>

- `df["col"].ffill()`
- `df["col"].bfill()`
- `df['col'].interpolate()`
<div dir='rtl'>

6. Multiple Imputation

---

### **06-03**
## Feature scaling
<div dir='rtl'>

دو تا از انواع معروفش :
</div>

- Normalization(Min-Max scaling) : کمترین مقدار را به ۰، بیشترین مقدار را به ۱، و هر مقدار دیگر را به عددی اعشاری بین ۰ و ۱ نگاشت می‌کند.
- Standardization : داده‌ها را حول عدد صفر متمرکز کرده و آن‌ها را به گونه‌ای مقیاس‌دهی می‌کند که دارای واریانس واحد باشند
<div dir='rtl'>

ویژگی های Normalization :
- دیگه مقدار صفر رو، صفر نشون نمیده برای همین برای دادهای sparse خوب نیست.
- خیلی حساس به داده های پرت. چون دقیقا min و max رو داره

نکته :
- زمانی که داده رو normaliz میکنی shap تغییری نمیکنه فقط فشرده میشه بین 0و1. (توزیعش عوض نمیشه)
</div>

- Other Data Scaling Methods    
  - Robust Scaling    
  - Max Abs Scaling   
  - Quantile Transformation    
  - Power Transformation

---

### **06-04**

# داده هایی که انگار طبیعی نیستند
<div dir='rtl'>

انواع داده های بی نظم:
1. **داده‌های پرت (Outliers)**: مقادیر حدی که تفاوت چشمگیری با سایر مشاهدات دارند.(یک نمونه منفرد و تنها)
2. **ناهنجاری‌ها (Anomalies)**: موارد یا الگوهای نادری که با رفتار مورد انتظار مطابقت ندارند.(یک دونه نیست)
3. **پدیده‌های نو (Novelties)**: الگوهای جدید یا پیش‌از‌این‌دیده‌نشده‌ای که در داده‌ها ظاهر می‌شوند.
4. **رانش داده (Data Drift)**: تغییرات تدریجی در ویژگی‌های آماری متغیر هدف در گذر زمان.(تغییرات منطقی هستند)

انواع داده های پرت:
1. **داده‌های پرت تک‌متغیره (Univariate Outliers):**
- در یک متغیر یا ویژگیِ واحد رخ می‌دهند. 
- مثال: در مجموعه داده‌ای شامل قد انسان‌ها، رکوردی که عدد ۳ متر را نشان دهد، یک داده پرت تک‌متغیره محسوب می‌شود.

2. **داده‌های پرت چندمتغیره (Multivariate Outliers):**
- در فضای n-بعدی که توسط چندین متغیر ایجاد شده است، رخ می‌دهند. 
- مثال: ترکیبی از سن و درآمد که بسیار غیرمعمول است؛ حتی اگر هر یک از این مقادیر به تنهایی داده پرت نباشند. فردی که 2 سالشه و درامدش 100هزاردلاره

3. **داده‌های پرت نقطه‌ای (Point Outliers):**
- نمونه‌های داده‌ای منفردی که از سایر داده‌ها بسیار دور هستند. 
- مثال: یک تراکنش مالی به مبلغ ۱,۰۰۰,۰۰۰ دلار در مجموعه داده‌ای که در آن بیشتر تراکنش‌ها کمتر از ۱,۰۰۰ دلار هستند.

4. **داده‌های پرت زمینه‌ای (Contextual Outliers):**
- نقاط داده‌ای که در یک زمینه (بافت) خاص، داده پرت محسوب می‌شوند، اما در شرایط دیگر چنین نیستند. 
- مثال: دمای ۳۰ درجه سانتی‌گراد ممکن است در تابستان طبیعی باشد، اما در داده‌های مربوط به دمای هوا در زمستان، یک داده پرت به شمار می‌آید.

برای شناسایی داده‌های پرت (Outliers) می‌توان از روش‌های متعددی استفاده کرد:

1. **روش‌های آماری**:
- نمره Z (Z-score): مشخص می‌کند که یک داده تا چه اندازه (بر حسب انحراف معیار) از میانگین فاصله دارد. 
- روش دامنه میان‌چارکی (IQR): داده‌های پرت را به عنوان نقاطی شناسایی می‌کند که کمتر از مقدار $Q1 - 1.5 \times IQR$ یا بیشتر از مقدار $Q3 + 1.5 \times IQR$ باشند.

2. **تکنیک‌های بصری‌سازی**:
- نمودار جعبه‌ای (Box Plot): گروه‌هایی از داده‌های عددی را از طریق چارک‌هایشان به صورت گرافیکی نمایش می‌دهد. 
- نمودار پراکندگی (Scatter Plot): برای شناسایی داده‌های پرت در داده‌های دوبعدی مفید است.

3. **روش‌های یادگیری ماشین**:
- جنگل ایزوله‌سازی (Isolation Forest): به جای تعیین ویژگی‌های نقاط عادی، ناهنجاری‌ها را جداسازی (ایزوله) می‌کند.
    - کدوم دیتا خیلی راحت از بقیه جدامیشه. میشمارد تا ببیند برای اینکه یک دیتا را از بقیه جداکند چندتا کات لازم دارد. هرکدوم که کات کمتری لازم داشت میشه داده پرت
- عامل داده پرت محلی Local Outlier Factor(LOF): چگالی محلی یک نقطه را با چگالی همسایگانش مقایسه می‌کند.(تذکر: globaly نیست.)

رویکرد برخورد با داده‌های پرت (Outliers) به ماهیت داده‌ها و اهداف تحلیل بستگی دارد:

1. **حذف**: اگر داده‌ی پرت ناشی از خطا باشد، حذف آن می‌تواند مناسب باشد.
2. **تبدیلTransform**: اعمال تبدیل‌هایی نظیر لگاریتم یا (Box-Cox) برای کاهش تأثیر مقادیر بسیار بزرگ یا کوچک.(اونایی که خیلی شلوغ میکنن رو محکم میزنی اما اونایی که ساکت ترند رو یواش تر میزنی)
3. **تحلیل جداگانه**: اگر داده‌های پرت نشان‌دهنده‌ی رویدادهای نادر اما مهم هستند، آن‌ها را به‌صورت جداگانه تحلیل کنید.(اونایی که خیلی شرن برن توی ی کلاس و اونایی که ساکتن برن توی ی کلاس دیگه)
4. **استفاده از روش‌های مقاوم (Robust)**: به‌کارگیری روش‌های آماری که نسبت به داده‌های پرت حساسیت کمتری دارند؛ برای مثال، استفاده از میانه به‌جای میانگین.(ایشالاه خود الگوریتم تربیتشون میکنه. میره توی جامعه درست میشه. سپردن دست خدا)
5. **Capping**: ماکزیموم و مینیموم دز نظر بگیر.


## Detecting anomalies:

1. **روش‌های آماری**:

- تحلیل مقادیر حدی: نقاط داده‌ای را که از توزیع آماری مجموعه داده‌ها منحرف می‌شوند، شناسایی می‌کند.

- باقیمانده‌های مدل رگرسیون: باقیمانده‌های(خطاهای) یک مدل رگرسیون را برای شناسایی نقاط غیرعادی تجزیه و تحلیل می‌کند.

2. **تکنیک‌های یادگیری ماشین**:

- روش‌های مبتنی بر خوشه‌بندی: نقاط داده‌ای را که به هیچ خوشه‌ای تعلق ندارند یا خوشه‌های بسیار کوچکی را تشکیل می‌دهند، شناسایی می‌کند.

- جنگل جداسازیIsolation Forest: ناهنجاری‌ها را با انتخاب تصادفی یک ویژگی و سپس انتخاب تصادفی یک مقدار تقسیم بین حداکثر و حداقل مقادیر آن ویژگی، جداسازی می‌کند.
    - داده هایی که کات کمتردارند رو میگه ناهنجاری

- ماشین بردار پشتیبان تک کلاسهOne-Class SVM: یک مرز تصمیم‌گیری را یاد می‌گیرد تا داده‌های جدید را به عنوان مشابه یا متفاوت با مجموعه آموزشی طبقه‌بندی کند.
    - همه رو سعی میکنه ببره توی یک کلاس؛ اگر نتونست یعنی اون داده ناهنجاره

3. **روش‌های خاص سری زمانی**:

- میانگین متحرک: ناهنجاری‌ها را با مقایسه هر نقطه با میانگین متحرک تشخیص می‌دهد.

- تجزیه فصلی: سری‌های زمانی را به روند، فصلی بودن و باقیمانده‌ها تجزیه می‌کند تا ناهنجاری‌ها را شناسایی کند.

Handling Anomalies:
1. **جداسازی**: روش: ایجاد دسته‌بندی‌های مجزا یا تعیین نشانگرهایی (Flags) برای نقاط داده‌ایِ دارای ناهنجاری.
2. **روش‌های ترکیبی (Ensemble Methods)**: روش: ترکیب چندین الگوریتم تشخیص ناهنجاری و استفاده از رأی‌گیری اکثریت یا میانگین‌گیری وزنی.

## برخی از ویژگی‌های کلیدی «پدیده‌های نو» (Novelties) :
1. **تازگی**: پدیده‌های نو بیانگر الگوهایی هستند که در داده‌های آموزشی(training) مشاهده نشده‌اند.
2. **احتمال تکرار**: برخلاف ناهنجاری‌ها، پدیده‌های نو ممکن است به‌مرور زمان تکرار بیشتری داشته باشند.
3. **ماهیت آگاهی‌بخش**: پدیده‌های نو اغلب بینش‌های ارزشمندی درباره روندها یا رفتارهای در حال تحول ارائه می‌دهند.
4. **وابستگی به حوزه**: اینکه چه چیزی یک «پدیده نو» محسوب می‌شود، می‌تواند بسته به حوزه یا کاربرد خاص، به‌طور قابل‌توجهی متفاوت باشد.

تشخیص Novelties:

1. **طبقه‌بندی تک‌کلاسه One-Class Classification**:

- روی داده‌های «عادی» آموزش می‌دهد و نمونه‌هایی را که با این الگو مطابقت ندارند، شناسایی می‌کند.
    - یک مرز تشکیل می دهد و هرکس داخل این مرز جانشد میشه ناهنجار

- مثال: SVM تک‌کلاسه

2. **تکنیک‌های خوشه‌بندی**:

- خوشه‌ها یا نقاط داده جدیدی را که در خوشه‌های موجود قرار نمی‌گیرند، شناسایی می‌کند.

- مثال: DBSCAN (خوشه‌بندی مکانی مبتنی بر چگالی برنامه‌ها با نویز)

3. **روش‌های یادگیری تطبیقی**:

- به‌روزرسانی مداوم مدل برای تشخیص الگوهای جدید.

- مثال: الگوریتم‌های یادگیری آنلاین

4. **کاهش ابعاد Dimensionality Reduction**:(فقط مخصوص novel نیست. روی همه کار میکنه)

- تجسم داده‌های با ابعاد بالا برای تشخیص الگوهای جدید.

- مثال: t-SNE (تعبیه همسایه تصادفی توزیع‌شده با t)

Handling Novelties:
1. **به‌روزرسانی مدل**: گنجاندن الگوهای جدید در مدل موجود از طریق آموزش مجدد یا یادگیری آنلاین.
2. **ایجاد دسته‌های جدید**: اگر موارد جدید نشان‌دهنده یک طبقه (کلاس) جدید و قابل‌توجه باشند، ایجاد یک دسته جدید در مدل.
3. **بررسی بیشتر**: تحلیل موارد جدید برای درک علت و پیامدهای آن‌ها.
4. **پایش**: رصد کردن بسامد و ماهیت موارد جدید به‌منظور شناسایی روندهای نوظهور.

## Handling Data Drift:

1. **بازآموزی منظم**:

- آموزش مجدد مدل بر اساس داده‌های اخیر به صورت دوره‌ای برای ثبت الگوهای جدید.


2. **یادگیری افزایشی**:

- استفاده از الگوریتم‌هایی که می‌توانند به طور مداوم از داده‌های جدید بدون فراموش کردن الگوهای قدیمی یاد بگیرند.


3. **روش‌های گروهی**:(ترکیب چند مدل)

- حفظ مجموعه‌ای از مدل‌های آموزش دیده در پنجره‌های زمانی مختلف برای ثبت توزیع‌های مختلف داده‌ها.
- زمان استفاده: هنگامی که ممکن است انواع مختلفی از «انحراف» (Drift) رخ دهد. 
- روش: نگهداری چندین مدل که بر روی دوره‌های زمانی متفاوت آموزش دیده‌اند و استفاده از مدلی که بهترین عملکرد را دارد.


4. **مهندسی ویژگی**:

- توسعه ویژگی‌های قوی که کمتر مستعد انحراف هستند.(دیتایی رو انتخاب کن که نخواد data drift داشته باشه تا دردسرها کم بشه باوجود اینکه مثلا دقت پایین بیاد)


5. **نرخ یادگیری تطبیقی**:

- تنظیم نرخ یادگیری بر اساس سطح انحراف شناسایی شده.

6. **Sliding Window**: همیشه تا 2 هفته یا 3 هفته رو نگاه کنید نه اینکه 2 سال پیش رو نگاه بکنید.

   ```python
   def sliding_window_train(model, X, y, window_size):
       for i in range(len(X) - window_size):
           X_window = X[i:i+window_size]
           y_window = y[i:i+window_size]
           model.fit(X_window, y_window)
   ```

7. **نمونه‌های وزن‌دار**:
- چه زمانی استفاده می‌شود: برای اختصاص اهمیت بیشتر به داده‌های اخیر، ضمن حفظ بخشی از اطلاعات قدیمی‌تر. 
- روش: در طول فرآیند آموزش، به نمونه‌های جدیدتر وزن‌های بالاتری اختصاص داده می‌شود.

---

### **06-05**

## Encoding
<div dir='rtl'>

برخی از روش ها:
</div>

- One-Hot Encoding
- Label Encoding : (برای یک ستون)
- Ordinal Encoding : (برای چنتا ستون)
- Target Encoding
- Frequency Encoding
- Binary Encoding : %pip install category_encoders
- Hashing Encoding
- Leave-One-Out Encoding
- Embedding Encoding
- Weight of Evidence (WOE) Encoding

---